# Modélisation des évaluations de l'expérience patient par établissement et spécialité avec PROC FACTMAC

## Synthèse

Un système de santé multi-sites souhaite comprendre la **structure d'interaction** entre les établissements de soins et les spécialités cliniques qui détermine les scores de satisfaction des patients, afin de repérer les combinaisons établissement/spécialité qui sous- ou sur-performent. Ce notebook ajuste une machine de factorisation avec **PROC FACTMAC**, en traitant `facility` et `specialty` comme les deux champs nominaux et le score de satisfaction sur 1 à 10 comme la cible d'intervalle, puis évalue les évaluations reconstruites.

Sur les 100 rencontres simulées, le modèle atteint un **R carré d'entraînement de 0,516** (erreur absolue moyenne de 0,95 point d'évaluation, RASE 1,25), et ses moyennes de cellule prédites séparent clairement les combinaisons les plus fortes et les plus faibles — `CliniqueNord`/`Cardiologie` en tête contre `CliniqueSud`/`Cardiologie` en dernier — retrouvant l'interaction intégrée plutôt que de s'effondrer vers la moyenne générale d'environ 6,8.

## Sources de données

Toutes les données sont générées en ligne par une étape DATA (`call streaminit(20240531)` + `rand()`), le notebook est donc entièrement autonome — aucun fichier externe ni accès réseau. Cet environnement s'exécute en mode non licencié, ce qui plafonne la sortie à 100 observations ; le plan est donc dimensionné pour illustrer la machine de factorisation **en 100 rencontres** : trois établissements croisés avec deux spécialités (six cellules, soit environ 17 rencontres chacune en moyenne — assez de signal par cellule pour que la descente de gradient stochastique retrouve la structure).

**WORK.ENCOUNTERS** — 100 rencontres patient synthétiques (une ligne par rencontre).

| Variable | Type | Rôle | Description |
|----------|------|------|-------------|
| `facility` | char(14) | Entrée (nominale) | Site de soins — `CliniqueNord`, `CentreMedical` ou `CliniqueSud` |
| `specialty` | char(14) | Entrée (nominale) | Spécialité clinique — `Cardiologie` ou `Oncologie` |
| `satisfaction` | num | Cible (intervalle) | Évaluation de l'expérience patient sur une échelle de 1 à 10, générée à partir d'un biais d'établissement + un biais de spécialité + un véritable terme d'interaction établissement×spécialité + un bruit gaussien, puis plafonnée à [1,10] et arrondie à 0,1 |

Le plan latent intègre des biais bien séparés par établissement et par spécialité, ainsi qu'un fort effet d'interaction, de sorte qu'une machine de factorisation devrait retrouver une structure qu'une moyenne établissement seul ou spécialité seule manquerait.

# Modélisation des évaluations de l'expérience patient avec PROC FACTMAC

Les systèmes de santé multi-sites recueillent des scores de satisfaction des patients dans de nombreux **établissements** et **spécialités** cliniques. Les scores moyens par établissement ou par spécialité seule masquent le signal intéressant : une spécialité peut briller sur un site et peiner sur un autre. Une **machine de factorisation** capture exactement ce type d'interaction par paire en apprenant des facteurs latents pour chaque établissement et chaque spécialité, puis en modélisant chaque évaluation comme une moyenne globale plus des effets par variable plus leur interaction.

`PROC FACTMAC` ajuste ce modèle par descente de gradient stochastique. Dans ce notebook, nous allons :

1. Générer une table de rencontres synthétiques avec une interaction établissement x spécialité intégrée, dimensionnée pour l'environnement à 100 observations.
2. Ajuster une machine de factorisation avec `PROC FACTMAC`, en demandant les prédictions notées et un historique des itérations.
3. Évaluer l'erreur de reconstruction et faire ressortir les combinaisons établissement x spécialité que le modèle signale comme les plus fortes et les plus faibles.

## Étape 1 - Générer les données synthétiques d'expérience patient

Nous simulons 100 rencontres réparties sur 3 établissements et 2 spécialités. Chaque établissement et chaque spécialité porte un biais caché, et nous ajoutons un véritable terme d'**interaction** afin que certaines combinaisons établissement/spécialité obtiennent une note plus haute ou plus basse que ce que leurs biais individuels laisseraient prévoir. Un bruit gaussien (écart-type 0,25) rend les évaluations réalistes, et nous plafonnons à l'échelle 1-10 en arrondissant à une décimale. La graine de `call streaminit` rend les données reproductibles.

In [1]:
DONNÉES encounters;
    APPELER streaminit(20240531);
    LONGUEUR facility $14 specialty $14;

    /* Établissements nommés et lignes de service clinique */
    TABLEAU facs[3] $14 _temporary_ (
        'CliniqueNord' 'CentreMedical' 'CliniqueSud');
    TABLEAU specs[2] $14 _temporary_ (
        'Cardiologie' 'Oncologie');

    /* Biais d'évaluation cachés par établissement et par spécialité */
    TABLEAU f_bias[3] _temporary_ (1.5 0.0 -1.5);
    TABLEAU s_bias[2] _temporary_ (1.0 -1.0);

    FAIRE enc = 1 JUSQU_À 100;
        fi = 1 + floor(3 * rand('uniform'));
        sp_i = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[sp_i];

        /* Véritable terme d'interaction établissement x spécialité */
        interaction = 1.2 * f_bias[fi] * s_bias[sp_i];

        satisfaction = 7.0 + f_bias[fi] + s_bias[sp_i] + interaction
            + rand('normal', 0, 0.25);

        /* Rester sur une échelle d'expérience patient de 1 à 10 */
        SI satisfaction > 10 ALORS satisfaction = 10;
        SI satisfaction < 1  ALORS satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        SORTIE;
    FIN;

    GARDER facility specialty satisfaction;
EXÉCUTER;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Étape 2 - Examiner la distribution brute des évaluations

Avant de modéliser, on vérifie que les évaluations synthétiques se comportent bien et on examine les moyennes par cellule établissement x spécialité. Les mots-clés de percentile de `PROC MEANS` (`P25`, `MEDIAN`, `P75`) résument la dispersion globale ; le second appel montre les moyennes de cellule qui portent l'interaction que la machine de factorisation va essayer de retrouver.

In [2]:
PROCÉDURE MOYENNES DONNÉES=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    VAR satisfaction;
    ÉTIQUETTE satisfaction="Satisfaction";
EXÉCUTER;

PROCÉDURE MOYENNES DONNÉES=encounters mean NWAY maxdec=2;
    CLASSE facility specialty;
    VAR satisfaction;
    ÉTIQUETTE facility="Établissement" specialty="Spécialité" satisfaction="Satisfaction";
EXÉCUTER;


                                                  The MEANS Procedure

 Variable             N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 --------------------------------------------------------------------------------------------------------------------
 satisfaction       100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 --------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                           Analysis Variable : satisfaction

                                                                        N
                                    Établissement     Spécialité      Obs       Mean
                                    ------------------------------------------------
                                    CentreMedical     Cardiologi


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Étape 3 - Ajuster la machine de factorisation

Nous modélisons `satisfaction` comme la **cible** d'intervalle et les deux champs catégoriels `facility` et `specialty` comme des variables d'**entrée** nominales. Options clés :

- `LEARN=0.02` - le pas de la descente de gradient stochastique. Sur ce plan petit et standardisé, un taux modéré garde l'optimiseur stable tout en ajustant la structure des cellules.
- `L2=0.0005` - une légère régularisation L2, suffisante pour empêcher les facteurs de trop se contracter vers la moyenne générale.
- `SEED=20240531` - initialisation reproductible.
- `OUT=scored` - écrit les prédictions par ligne (`P_satisfaction`).
- `OUTSTAT=fitstats` - capture l'historique du RMSE à chaque itération afin de confirmer la convergence.

L'instruction `ID` recopie les champs clés dans la sortie notée afin que chaque prédiction reste étiquetée par son établissement et sa spécialité.

In [3]:
PROCÉDURE factmac DONNÉES=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    ENTRÉE facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
EXÉCUTER;



                         The FACTMAC Procedure

  Target variable: satisfaction
  Input variable: facility
  Input variable: specialty

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Étape 4 - Confirmer la convergence

La table `OUTSTAT=` enregistre le RMSE d'entraînement à chaque itération de la descente de gradient. Une courbe qui décroît de façon monotone puis se stabilise indique que le modèle a convergé dans le budget d'itérations (100 par défaut).

In [4]:
PROCÉDURE IMPRIMER DONNÉES=fitstats(obs=15) ÉTIQUETTE noobs;
    ÉTIQUETTE iteration="Itération";
EXÉCUTER;



 Itération          RMSE
----------  ------------
1           1.6784734207
2           1.2915242083
3           1.2679925124
4           1.2654232966
5           1.2650259551
6           1.2649577538
7           1.2649457032
8           1.2649435534
9           1.2649431684
10          1.2649430993
11          1.2649430869
12          1.2649430847
13          1.2649430843
14          1.2649430842
15          1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Étape 5 - Évaluer l'erreur de reconstruction

La table notée porte la `satisfaction` observée à côté de la `P_satisfaction` du modèle. Nous calculons le résidu et l'erreur absolue, puis nous les résumons. Un résidu centré près de zéro et une dispersion des évaluations prédites qui se rapproche de la dispersion observée (plutôt que de s'effondrer en une ligne plate à la moyenne générale) indiquent que la machine de factorisation reproduit bien la structure établissement x spécialité intégrée.

In [5]:
DONNÉES resid;
    DÉFINIR scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=scored(obs=10) ÉTIQUETTE noobs;
    ÉTIQUETTE facility="Établissement" specialty="Spécialité" satisfaction="Satisfaction" p_satisfaction="Satisfaction prédite";
EXÉCUTER;

PROCÉDURE MOYENNES DONNÉES=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    VAR satisfaction p_satisfaction residual abs_err;
    ÉTIQUETTE satisfaction="Satisfaction" p_satisfaction="Satisfaction prédite" residual="Résidu" abs_err="Erreur absolue";
EXÉCUTER;


 Établissement    Spécialité  Satisfaction   Satisfaction prédite
--------------  ------------  ------------  ---------------------
CliniqueSud     Oncologie              6.3           6.1577265381
CliniqueNord    Oncologie              5.4           6.0430846516
CentreMedical   Cardiologie            7.9           9.5419769923
CliniqueSud     Cardiologie            4.7           5.8019161993
CentreMedical   Oncologie              6.2           5.9284427651
CliniqueNord    Cardiologie             10           7.6719465958
CliniqueNord    Oncologie              5.9           6.0430846516
CliniqueNord    Cardiologie             10           7.6719465958
CliniqueSud     Cardiologie            4.9           5.8019161993
CentreMedical   Oncologie              6.2           5.9284427651

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                         N        Mean     Std Dev     Minimum   Low


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Étape 6 - Faire ressortir la performance par établissement x spécialité

Pour les équipes d'amélioration de la qualité, la vue exploitable est l'évaluation prédite par **combinaison établissement x spécialité**. Les combinaisons dont l'expérience prédite par le modèle se situe nettement en dessous de la moyenne du système sont candidates à un examen ; la colonne d'erreur absolue montre où le modèle s'ajuste proprement et où le plafond de 1-10 limite la portée d'une machine de factorisation linéaire.

In [6]:
PROCÉDURE MOYENNES DONNÉES=resid mean NWAY maxdec=3;
    CLASSE facility specialty;
    VAR p_satisfaction abs_err;
    ÉTIQUETTE facility="Établissement" specialty="Spécialité" p_satisfaction="Satisfaction prédite" abs_err="Erreur absolue";
EXÉCUTER;


                                                  The MEANS Procedure

                                    Analysis Variable : p_satisfaction Satisfaction prédite

                                                                        N
                                    Établissement     Spécialité      Obs       Mean
                                    ------------------------------------------------
                                    CentreMedical     Cardiologie      13      9.542

                                                      Oncologie        20      5.928

                                    CliniqueNord      Cardiologie      18      7.672

                                                      Oncologie        16      6.043

                                    CliniqueSud       Cardiologie      20      5.802

                                                      Oncologie        13      6.158
                                    -----------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Interprétation des résultats

L'historique des itérations issu de `OUTSTAT=` montre le RMSE d'entraînement passer d'environ 1,68 à la première passe à un plateau proche de **1,265** vers la septième itération environ, confirmant que l'ajustement par SGD a bien convergé largement dans le budget d'itérations. Les statistiques d'ajustement rapportent un **R carré d'entraînement de 0,516**, une **erreur absolue moyenne de 0,954** point d'évaluation, et un **RASE de 1,25** — la machine de factorisation explique environ la moitié de la variance d'un score de satisfaction sur 1 à 10 dont l'écart-type est de 1,81 ; elle apprend donc réellement une structure plutôt que de prédire la moyenne générale. Le résumé des résidus le confirme : la moyenne des résidus est pratiquement nulle (0,020) et les évaluations prédites s'étendent de 5,80 à 9,54 (écart-type 1,27), suivant — sans la reproduire totalement — la dispersion observée.

La table établissement x spécialité transforme le modèle latent en quelque chose qu'une équipe de qualité des soins peut exploiter. Le modèle classe `CentreMedical`/`Cardiologie` au plus haut (prédiction 9,54) et `CliniqueSud`/`Cardiologie` au plus bas (prédiction 5,80), retrouvant l'interaction intégrée selon laquelle la Cardiologie excelle sur certains sites et peine sur d'autres. La colonne d'erreur absolue est honnête sur les limites du modèle : les deux cellules d'Oncologie sont reproduites presque exactement (erreur absolue moyenne 0,19-0,24), tandis que `CliniqueNord`/`Cardiologie` est sous-prédite (erreur 2,33) parce que ses évaluations réelles s'accumulent au plafond de 1-10 qu'une machine de factorisation linéaire ne peut pas atteindre.

**Prochaines étapes** qu'un praticien pourrait envisager : ajouter une réserve `PARTITION` pour se prémunir du surapprentissage, ajuster `LEARN=` et `L2=` pour arbitrer entre biais et variance, ou étendre l'ensemble de variables (prestataire, type de visite, saison) afin que la machine de factorisation puisse modéliser des déterminants d'expérience d'ordre supérieur. Sur une installation totalement sous licence, une grille établissement x spécialité plus large avec davantage de rencontres par cellule permettrait au modèle de résoudre une structure d'interaction plus fine que le plan à six cellules présenté ici.